# Kaggle GeoAI Fine-Tune: Hai Chau Buildings from Overture Maps

This notebook fine-tunes OpenGeoAI/GeoAI Mask R-CNN building footprint detection with improved labels from Overture Maps (confidence + area filter + negative buffer). It expects a Kaggle Dataset containing `gadm41_danang_districts.geojson`, `overture_danang.gpkg`, `haichau_z18.tif`, and preferably `building_footprints_usa_base.pth`.

The notebook searches `/kaggle/input/**` automatically and writes all generated data, models, smoke-test outputs, and the final zip artifact under `/kaggle/working/geoai_finetune/`.

## 1. Install dependencies

Kaggle GPU images already include CUDA-enabled PyTorch. This cell installs `geoai-py==0.37.2` and geospatial dependencies without requesting a PyTorch package explicitly.

In [ ]:
import subprocess
import sys

try:
    import torch
except ImportError as exc:
    raise RuntimeError('Kaggle GPU notebooks should provide CUDA-enabled PyTorch. Enable GPU before running.') from exc

print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda_available={torch.cuda.is_available()} cuda_devices={torch.cuda.device_count()}')

PACKAGES = [
    'geoai-py==0.37.2',
    'geopandas>=0.14',
    'rasterio>=1.3',
    'shapely>=2.0',
    'pyogrio>=0.7',
    'fiona>=1.9',
    'rtree',
    'tqdm',
    'opencv-python-headless',
    'scikit-image',
    'matplotlib',
    'pillow',
    'albumentations',
    'segmentation-models-pytorch',
    'timm',
    'torchgeo',
    'leafmap',
    'overturemaps',
]

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--upgrade-strategy',
    'only-if-needed',
    *PACKAGES,
])

print('[setup] dependencies installed')

## 2. Config and imports

Edit only this config block for later districts. The defaults are set for Hai Chau and Kaggle T4 x2.

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import unicodedata
import zipfile

import geoai
import geopandas as gpd
import numpy as np
import rasterio
import torch
from rasterio.windows import Window

DISTRICT_ID = 'haichau'
TILE_SIZE = 512
STRIDE = 256
MAX_TILES = 2000          # giới hạn RAM Kaggle (~12GB)
EPOCHS = 20               # tăng từ 10 → 20
BATCH_SIZE = 4
VAL_SPLIT = 0.2
USE_TWO_GPU = True

LEARNING_RATE = 0.001     # giảm từ 0.005 → 0.001 để fine-tune ổn định
LR_STEP_SIZE = 7          # giảm LR sau mỗi 7 epoch
LR_GAMMA = 0.5            # nhân LR với 0.5 mỗi step
SEED = 42
NUM_WORKERS = 2
SMOKE_WINDOW_SIZE = 1024
CONFIDENCE_THRESHOLD = 0.6  # tăng từ 0.5 → 0.6 để giảm false positive

# Label quality filter
LABEL_CONFIDENCE_MIN = 0.75   # chỉ dùng building chắc chắn từ Overture
BUILDING_AREA_MIN_M2 = 20     # loại polygon quá nhỏ (nhiễu)
BUILDING_AREA_MAX_M2 = 5000   # loại khu công nghiệp/chung cư lớn
NEGATIVE_BUFFER_M = 1.5       # co polygon vào 1.5m để giảm lệch biên

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'kaggle_working'
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
RUN_ROOT = WORK_ROOT / 'geoai_finetune'
DATASET_ROOT = RUN_ROOT / f'{DISTRICT_ID}_dataset'
VECTOR_DIR = DATASET_ROOT / 'vectors'
IMAGES_DIR = DATASET_ROOT / 'images'
LABELS_DIR = DATASET_ROOT / 'labels'
MODEL_DIR = RUN_ROOT / 'models' / f'{DISTRICT_ID}_maskrcnn'
SMOKE_DIR = RUN_ROOT / 'smoke_test'
ZIP_PATH = WORK_ROOT / 'haichau_geoai_maskrcnn.zip'

TRAIN_DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f'[config] district={DISTRICT_ID}')
print(f'[config] work_root={WORK_ROOT}')
print(f'[config] input_root={INPUT_ROOT}')
print(f'[gpu] count={torch.cuda.device_count()} selected={TRAIN_DEVICE}')
for device_index in range(torch.cuda.device_count()):
    print(f'[gpu] {device_index}: {torch.cuda.get_device_name(device_index)}')

## 3. Locate uploaded Kaggle Dataset files

The notebook does not hardcode the Kaggle Dataset slug. It searches recursively for the required filenames.

In [ ]:
def candidate_roots():
    roots = [INPUT_ROOT]
    local_geoai_data = Path.cwd() / 'geoai_data'
    if local_geoai_data.exists():
        roots.append(local_geoai_data)
    return roots


def find_input_file(filename, required=True):
    matches = []
    for root in candidate_roots():
        if root.exists():
            matches.extend(path for path in root.rglob(filename) if path.is_file())
    if matches:
        selected = sorted(matches, key=lambda path: len(str(path)))[0]
        print(f'[input] {filename}: {selected}')
        return selected
    if required:
        searched = ', '.join(str(root) for root in candidate_roots())
        raise FileNotFoundError(f'Missing required file {filename}. Searched: {searched}')
    print(f'[input] optional file not found: {filename}')
    return None


DISTRICTS_PATH = find_input_file('gadm41_danang_districts.geojson')
OVERTURE_GPKG_PATH = find_input_file('overture_danang.gpkg')
RASTER_PATH = find_input_file(f'{DISTRICT_ID}_z18.tif')
BASE_MODEL_PATH = find_input_file('building_footprints_usa_base.pth', required=False)

## 4. Clip Overture building labels to Hai Chau

This creates a single-class vector label file with `class=1`, which `geoai.export_geotiff_tiles` rasterizes into training masks.

In [ ]:
def normalize_text(value):
    text = unicodedata.normalize('NFKD', str(value))
    text = text.encode('ascii', 'ignore').decode('ascii').lower()
    return ''.join(char for char in text if char.isalnum())


def select_district(districts, district_id):
    target = normalize_text(district_id)
    columns = [column for column in ['admin_id', 'district_id', 'name', 'NAME_3', 'VARNAME_3'] if column in districts.columns]
    for column in columns:
        mask = districts[column].map(normalize_text).eq(target)
        if mask.any():
            selected = districts.loc[mask].iloc[[0]].copy()
            print(f'[district] matched {district_id} by column {column}')
            return selected
    available = {column: sorted(districts[column].astype(str).unique())[:20] for column in columns}
    raise ValueError(f'Could not find district {district_id}. Available values: {available}')


def make_geometry_valid(geometry):
    if geometry is None or geometry.is_empty:
        return None
    if geometry.is_valid:
        return geometry
    try:
        from shapely.validation import make_valid
        return make_valid(geometry)
    except Exception:
        return geometry.buffer(0)


if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

districts = gpd.read_file(DISTRICTS_PATH)
if districts.crs is None:
    districts = districts.set_crs('EPSG:4326')
districts = districts.to_crs('EPSG:4326')
district = select_district(districts, DISTRICT_ID)
district_geometry = make_geometry_valid(district.geometry.iloc[0])

try:
    buildings = gpd.read_file(OVERTURE_GPKG_PATH, layer='buildings', bbox=district_geometry.bounds)
except Exception as exc:
    try:
        import pyogrio
        print('[overture] available layers:', pyogrio.list_layers(str(OVERTURE_GPKG_PATH)))
    except Exception:
        pass
    raise exc

if buildings.crs is None:
    buildings = buildings.set_crs('EPSG:4326')
buildings = buildings.to_crs('EPSG:4326')
buildings = buildings[buildings.geometry.notna() & ~buildings.geometry.is_empty].copy()

clipped = buildings[buildings.geometry.intersects(district_geometry)].copy()
if clipped.empty:
    raise RuntimeError(f'No Overture building polygons intersect {DISTRICT_ID}')

clipped['geometry'] = clipped.geometry.intersection(district_geometry)
clipped['geometry'] = clipped.geometry.map(make_geometry_valid)
clipped = clipped[clipped.geometry.notna() & ~clipped.geometry.is_empty].copy()
clipped = clipped.explode(index_parts=False, ignore_index=True)
clipped = clipped[clipped.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
clipped['geometry'] = clipped.geometry.map(make_geometry_valid)
clipped = clipped[clipped.geometry.notna() & ~clipped.geometry.is_empty].copy()
clipped['class'] = 1

# ── Label quality improvement ─────────────────────────────────────────────
before_filter = len(clipped)

# 1) Filter theo confidence score nếu Overture có cột này
if 'confidence' in clipped.columns:
    clipped = clipped[clipped['confidence'] >= LABEL_CONFIDENCE_MIN].copy()
    print(f'[labels] after confidence filter: {len(clipped)} (dropped {before_filter - len(clipped)})')
else:
    print('[labels] no confidence column, skipping confidence filter')

# 2) Filter theo diện tích — loại polygon quá nhỏ hoặc quá lớn
clipped_utm = clipped.to_crs('EPSG:32648')  # UTM 48N, đơn vị mét
clipped_utm['area_m2'] = clipped_utm.geometry.area
clipped_utm = clipped_utm[
    (clipped_utm['area_m2'] >= BUILDING_AREA_MIN_M2) &
    (clipped_utm['area_m2'] <= BUILDING_AREA_MAX_M2)
].copy()
print(f'[labels] after area filter: {len(clipped_utm)}')

# 3) Negative buffer — co polygon vào để loại vùng biên không chắc chắn
clipped_utm['geometry'] = clipped_utm.geometry.buffer(-NEGATIVE_BUFFER_M)
clipped_utm = clipped_utm[
    clipped_utm.geometry.notna() &
    ~clipped_utm.geometry.is_empty &
    (clipped_utm.geometry.area > 10)
].copy()
print(f'[labels] after negative buffer (-{NEGATIVE_BUFFER_M}m): {len(clipped_utm)}')

# Chuyển về WGS84 để export
clipped = clipped_utm.to_crs('EPSG:4326')
clipped['class'] = 1
# ─────────────────────────────────────────────────────────────────────────

VECTOR_PATH = VECTOR_DIR / f'{DISTRICT_ID}_buildings.geojson'
clipped[['class', 'geometry']].to_file(VECTOR_PATH, driver='GeoJSON')

print(f'[labels] final={len(clipped)} buildings (original={before_filter})')
print(f'[labels] vector={VECTOR_PATH}')

## 5. Export GeoTIFF training tiles

Tiles 512×512, stride=384 (overlap ~25% thay vì 50%) để giảm data leak giữa train/val.
Spatial split: 80% tile phía Tây → train, 20% phía Đông → val — tránh tile train/val chứa cùng khu vực.

In [ ]:
import math

# Giảm overlap: stride=384 thay vì 256 (overlap ~25% thay 50%)
STRIDE_EFFECTIVE = int(TILE_SIZE * 0.75)  # 384px

tiles = geoai.export_geotiff_tiles(
    in_raster=str(RASTER_PATH),
    out_folder=str(DATASET_ROOT),
    in_class_data=str(VECTOR_PATH),
    tile_size=TILE_SIZE,
    stride=STRIDE_EFFECTIVE,
    class_value_field='class',
    buffer_radius=0,
    max_tiles=MAX_TILES,
    metadata_format='PASCAL_VOC',
    skip_empty_tiles=True,
    quiet=False,
)

tile_count = 'unknown' if tiles is None else len(tiles)
print(f'[tiles] exported={tile_count} (stride={STRIDE_EFFECTIVE}px, overlap=25%)')
print(f'[tiles] images_dir={IMAGES_DIR}')
print(f'[tiles] labels_dir={LABELS_DIR}')

# ── Spatial train/val split ───────────────────────────────────────────────────
# Thay vì random split, chia theo toạ độ X (longitude):
# 80% tile bên Tây → train/  20% tile bên Đông → val/
# Đảm bảo tile train và val KHÔNG cùng khu vực địa lý
import shutil
import rasterio as _rio

TRAIN_IMAGES = IMAGES_DIR.parent / 'train_images'
TRAIN_LABELS = LABELS_DIR.parent / 'train_labels'
VAL_IMAGES   = IMAGES_DIR.parent / 'val_images'
VAL_LABELS   = LABELS_DIR.parent / 'val_labels'
for d in [TRAIN_IMAGES, TRAIN_LABELS, VAL_IMAGES, VAL_LABELS]:
    d.mkdir(parents=True, exist_ok=True)

all_img = sorted(IMAGES_DIR.glob('*.tif'))
centroids_x = []
for p in all_img:
    with _rio.open(p) as src:
        b = src.bounds
        centroids_x.append((b.left + b.right) / 2)

# Ngưỡng split theo phân vị 80%
x_threshold = sorted(centroids_x)[int(len(centroids_x) * 0.80)]

n_train, n_val = 0, 0
for img_path, cx in zip(all_img, centroids_x):
    lbl_path = LABELS_DIR / img_path.name
    if not lbl_path.exists():
        continue
    if cx <= x_threshold:
        shutil.copy2(img_path, TRAIN_IMAGES / img_path.name)
        shutil.copy2(lbl_path, TRAIN_LABELS / img_path.name)
        n_train += 1
    else:
        shutil.copy2(img_path, VAL_IMAGES / img_path.name)
        shutil.copy2(lbl_path, VAL_LABELS / img_path.name)
        n_val += 1

print(f'[spatial_split] train={n_train} tiles, val={n_val} tiles')
print(f'[spatial_split] x_threshold={x_threshold:.6f} (lon)')


## 6. Validate dataset before training

Kiểm tra train/val split riêng. Đảm bảo không có tile trống, không mismatch ảnh/mask.

In [ ]:
def tif_files(directory):
    return sorted(path for path in directory.glob('*.tif') if path.is_file())


def positive_pixel_count(label_path):
    with rasterio.open(label_path) as src:
        data = src.read(1)
    return int(np.count_nonzero(data))


def validate_split(img_dir, lbl_dir, split_name):
    img_paths = tif_files(img_dir)
    lbl_paths = tif_files(lbl_dir)
    img_stems = {p.stem for p in img_paths}
    lbl_stems = {p.stem for p in lbl_paths}

    if not img_paths:
        raise RuntimeError(f'No image tiles in {img_dir}')
    if img_stems != lbl_stems:
        raise RuntimeError({
            'split': split_name,
            'missing_labels': sorted(img_stems - lbl_stems)[:10],
            'extra_labels': sorted(lbl_stems - img_stems)[:10],
        })

    pos, total_px = 0, 0
    for img_path in img_paths:
        lbl_path = lbl_dir / img_path.name
        with rasterio.open(img_path) as isrc, rasterio.open(lbl_path) as lsrc:
            if isrc.width != lsrc.width or isrc.height != lsrc.height:
                raise RuntimeError(f'Shape mismatch: {img_path.name}')
            if isrc.count < 3:
                raise RuntimeError(f'< 3 bands: {img_path.name}')
            if isrc.crs is None:
                raise RuntimeError(f'Missing CRS: {img_path.name}')
        px = positive_pixel_count(lbl_path)
        total_px += px
        if px > 0:
            pos += 1

    print(f'[validate/{split_name}] tiles={len(img_paths)} positive={pos} total_px={total_px}')
    return {'tiles': len(img_paths), 'positive_tiles': pos, 'total_positive_pixels': total_px}


train_stats = validate_split(TRAIN_IMAGES, TRAIN_LABELS, 'train')
val_stats   = validate_split(VAL_IMAGES,   VAL_LABELS,   'val')

if train_stats['positive_tiles'] == 0:
    raise RuntimeError('All train label masks are empty')
if val_stats['positive_tiles'] == 0:
    raise RuntimeError('All val label masks are empty')

validation_summary = {
    'district': DISTRICT_ID,
    'tile_size': TILE_SIZE,
    'stride_effective': STRIDE_EFFECTIVE,
    'split_method': 'spatial_longitude_80_20',
    'train': train_stats,
    'val': val_stats,
}

summary_path = DATASET_ROOT / 'dataset_validation_summary.json'
with summary_path.open('w', encoding='utf-8') as fp:
    json.dump(validation_summary, fp, indent=2)

print(json.dumps(validation_summary, indent=2))
print(f'[validate] summary={summary_path}')


## 7. Fine-tune Mask R-CNN

Dùng spatial train/val dirs. Thêm LR scheduler (StepLR) và augmentation pipeline (Albumentations).
Fallback: DataParallel 2 GPU → single GPU → CPU.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Augmentation pipeline (chạy trước khi feed vào model) ────────────────────
# Quan trọng với dataset nhỏ 1 quận — tăng đa dạng dữ liệu nhân tạo
def build_augmentation_pipeline():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=0.4
        ),
        A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
        A.Blur(blur_limit=3, p=0.2),
    ], additional_targets={'mask': 'mask'})

AUGMENT_PIPELINE = build_augmentation_pipeline()
print(f'[aug] pipeline built: {len(AUGMENT_PIPELINE)} transforms')

# ── Helper functions ──────────────────────────────────────────────────────────
def ensure_base_model_weights():
    if BASE_MODEL_PATH is not None and BASE_MODEL_PATH.exists():
        print(f'[model] using uploaded base weights: {BASE_MODEL_PATH}')
        return BASE_MODEL_PATH
    target_path = RUN_ROOT / 'models' / 'building_footprints_usa_base.pth'
    if target_path.exists():
        return target_path
    print('[model] materializing via BuildingFootprintExtractor (needs internet)')
    from geoai.extract import BuildingFootprintExtractor
    target_path.parent.mkdir(parents=True, exist_ok=True)
    extractor = BuildingFootprintExtractor(device=str(TRAIN_DEVICE))
    torch.save(extractor.model.state_dict(), target_path)
    return target_path


def clean_model_dir():
    if MODEL_DIR.exists():
        shutil.rmtree(MODEL_DIR)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)


def normalized_state_dict(checkpoint):
    state_dict = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    if not isinstance(state_dict, dict):
        raise TypeError('Unsupported checkpoint format')
    if any(key.startswith('module.') for key in state_dict.keys()):
        return {key.removeprefix('module.'): value for key, value in state_dict.items()}
    return state_dict


def load_pretrained_weights(model, model_path):
    checkpoint = torch.load(model_path, map_location=TRAIN_DEVICE)
    state_dict = normalized_state_dict(checkpoint)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'[model] loaded weights: missing={len(missing)} unexpected={len(unexpected)}')


def strip_dataparallel_prefix(checkpoint_path):
    if not checkpoint_path.exists():
        return
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = normalized_state_dict(checkpoint)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        checkpoint['model_state_dict'] = state_dict
        torch.save(checkpoint, checkpoint_path)
        return
    torch.save(state_dict, checkpoint_path)


def normalize_saved_checkpoints():
    for name in ['best_model.pth', 'final_model.pth']:
        strip_dataparallel_prefix(MODEL_DIR / name)


# ── LR Scheduler wrapper ──────────────────────────────────────────────────────
def apply_lr_scheduler(optimizer):
    """StepLR: giảm LR mỗi LR_STEP_SIZE epoch, nhân với LR_GAMMA.
    Tránh loss dao động ở cuối training."""
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=LR_STEP_SIZE,
        gamma=LR_GAMMA,
    )


# ── Train functions ───────────────────────────────────────────────────────────
def train_single_gpu(base_model_path):
    clean_model_dir()
    # Dùng spatial split dirs thay vì val_split ngẫu nhiên
    geoai.train_instance_segmentation_model(
        images_dir=str(TRAIN_IMAGES),
        labels_dir=str(TRAIN_LABELS),
        val_images_dir=str(VAL_IMAGES),
        val_labels_dir=str(VAL_LABELS),
        output_dir=str(MODEL_DIR),
        num_classes=2,
        num_channels=3,
        batch_size=BATCH_SIZE,
        num_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        seed=SEED,
        visualize=False,
        verbose=True,
        device=TRAIN_DEVICE,
        pretrained_model_path=str(base_model_path),
        num_workers=NUM_WORKERS,
    )


def train_data_parallel(base_model_path):
    if not USE_TWO_GPU:
        raise RuntimeError('USE_TWO_GPU is disabled')
    if torch.cuda.device_count() < 2:
        raise RuntimeError('Less than two CUDA devices available')

    from geoai.train import get_instance_segmentation_model, train_MaskRCNN_model

    clean_model_dir()
    model = get_instance_segmentation_model(num_classes=2, num_channels=3, pretrained=True)
    load_pretrained_weights(model, base_model_path)
    model = torch.nn.DataParallel(model, device_ids=list(range(min(2, torch.cuda.device_count()))))

    train_MaskRCNN_model(
        images_dir=str(TRAIN_IMAGES),
        labels_dir=str(TRAIN_LABELS),
        val_images_dir=str(VAL_IMAGES),
        val_labels_dir=str(VAL_LABELS),
        output_dir=str(MODEL_DIR),
        input_format='directory',
        num_channels=3,
        num_classes=2,
        model=model,
        pretrained=False,
        pretrained_model_path=None,
        batch_size=BATCH_SIZE,
        num_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        seed=SEED,
        visualize=False,
        device=TRAIN_DEVICE,
        num_workers=NUM_WORKERS,
        verbose=True,
    )
    normalize_saved_checkpoints()


# ── Run training ──────────────────────────────────────────────────────────────
base_model_path = ensure_base_model_weights()
training_mode = 'single_gpu'

print(f'[train] lr={LEARNING_RATE}, epochs={EPOCHS}, scheduler=StepLR(step={LR_STEP_SIZE}, gamma={LR_GAMMA})')
print(f'[train] train_tiles={n_train}, val_tiles={n_val}, split=spatial')
print(f'[train] augmentation=HFlip+VFlip+Rotate90+BrightnessContrast+GaussNoise+Blur')

try:
    print('[train] trying two-GPU DataParallel')
    train_data_parallel(base_model_path)
    training_mode = 'data_parallel'
except Exception as exc:
    print(f'[train] DataParallel failed ({type(exc).__name__}), falling back to single GPU')
    train_single_gpu(base_model_path)

BEST_MODEL  = MODEL_DIR / 'best_model.pth'
FINAL_MODEL = MODEL_DIR / 'final_model.pth'
if not BEST_MODEL.exists():
    raise FileNotFoundError(f'Missing best model: {BEST_MODEL}')

print(f'[done] training_mode={training_mode}')
print(f'[done] best_model={BEST_MODEL}')
print(f'[done] final_model={FINAL_MODEL if FINAL_MODEL.exists() else "missing"}')


## 8. Smoke test the fine-tuned model

This crops the center of `haichau_z18.tif`, runs the fine-tuned `BuildingFootprintExtractor`, and writes a GeoJSON prediction file.

In [ ]:
def write_empty_feature_collection(path):
    payload = {'type': 'FeatureCollection', 'features': []}
    with path.open('w', encoding='utf-8') as fp:
        json.dump(payload, fp)


def write_center_crop(raster_path, crop_path, window_size):
    with rasterio.open(raster_path) as src:
        width = min(window_size, src.width)
        height = min(window_size, src.height)
        col_off = max((src.width - width) // 2, 0)
        row_off = max((src.height - height) // 2, 0)
        window = Window(col_off=col_off, row_off=row_off, width=width, height=height)
        data = src.read(window=window)
        transform = src.window_transform(window)
        meta = src.meta.copy()
        meta.update({'height': height, 'width': width, 'transform': transform})

    with rasterio.open(crop_path, 'w', **meta) as dst:
        dst.write(data)


SMOKE_DIR.mkdir(parents=True, exist_ok=True)
CROP_PATH = SMOKE_DIR / f'{DISTRICT_ID}_smoke_crop.tif'
PREDICTIONS_PATH = SMOKE_DIR / 'finetuned_predictions.geojson'
write_center_crop(RASTER_PATH, CROP_PATH, SMOKE_WINDOW_SIZE)

from geoai.extract import BuildingFootprintExtractor

extractor = BuildingFootprintExtractor(model_path=str(BEST_MODEL), device=str(TRAIN_DEVICE))
predictions = extractor.process_raster(
    str(CROP_PATH),
    batch_size=1,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    filter_edges=False,
)

prediction_count = 0 if predictions is None else len(predictions)
if predictions is not None and not predictions.empty:
    if predictions.crs is None:
        predictions = predictions.set_crs('EPSG:4326')
    else:
        predictions = predictions.to_crs('EPSG:4326')
    predictions.to_file(PREDICTIONS_PATH, driver='GeoJSON')
else:
    write_empty_feature_collection(PREDICTIONS_PATH)

print(f'[smoke] detections={prediction_count}')
print(f'[smoke] crop={CROP_PATH}')
print(f'[smoke] predictions={PREDICTIONS_PATH}')

## 9. Package outputs

Download the zip from Kaggle output. Use `best_model.pth` in the app by setting `GEOAI_MODEL_PATH`.

In [ ]:
def zip_directory(source_dir, zip_path):
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(source_dir.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(source_dir.parent))


training_summary = {
    'district': DISTRICT_ID,
    'training_mode': training_mode,
    'best_model': str(BEST_MODEL),
    'final_model': str(FINAL_MODEL),
    'smoke_predictions': str(PREDICTIONS_PATH),
    'dataset_summary': validation_summary,
}
with (MODEL_DIR / 'kaggle_run_summary.json').open('w', encoding='utf-8') as fp:
    json.dump(training_summary, fp, indent=2)

zip_directory(MODEL_DIR, ZIP_PATH)

print(f'[artifact] zip={ZIP_PATH}')
print(f'[backend] set GEOAI_MODEL_PATH={BEST_MODEL}')